# Qwen3-VL Shared Domain Base — Overnight Training (all non-Stage-4 VLM stages)

**Decision this implements:** Molmo2 (512+enhanced tiling) owns Stage 4 detection. **Qwen3-VL-8B
becomes the shared domain base for every other VLM stage** (1 sheet-classification, 2 title-block,
5 OCR-tiebreaker, 10.5 skid grouping, 12 relation validation, 13 entity validation). This is a
recorded deviation from the spec's one-base rule (`PID_Local_Substitution_Spec.md` §5) — the spec
itself allows it when the Stage-4 winner doesn't fit the reasoning stages.

**Task-neutral by design (CLAUDE.md rule 3):** trains P&ID domain knowledge (tag grammar, symbol
vocabulary, layout conventions, connectivity), NOT any stage's output format. Stage-specific
behavior comes later via prompts (or per-stage LoRA only on measured failure).

**Four mixed tasks:**
1. **OCR tag-reading** (Gupta, Tesseract pseudo-labels — machine-generated, not verified GT)
2. **Symbol counting** (Gupta real boxes, class-agnostic)
3. **Typed symbol summary** (Kaggle synthetic, 32 classes)
4. **Connectivity Q&A** (PID2Graph OPEN100 real graph edges) — *new*: "are these two symbols
   directly connected?" — the only real ground truth for what stages 10.5/12 actually do.

**Overnight resilience — how this survives you being asleep:**
- All data pulled from HF Hub (private repo `timthy45/pnid-extraction-datasets`) to **local VM
  disk** — zero Google Drive, zero auth popups, fast reads.
- Checkpoint every 200 steps → saved locally AND pushed to a private HF model repo
  (`timthy45/qwen3vl-pnid-domain-base`). If the VM dies at 3am, nothing is lost.
- On any rerun, the notebook **auto-resumes** from the latest HF checkpoint (exact epoch+step).
- Per-step exception handling: a corrupt image or OOM skips the step instead of killing the run.
- Time budget: stops cleanly with a final checkpoint before Colab's ~24h limit can kill it.

**⚠️ To actually run unattended you must do one thing:** Colab Pro+ → toolbar → Runtime →
*"Run cell... in background"* / enable **background execution** when starting Run All. Pro+ keeps
executing after you close the browser — that's the feature this notebook is designed around.
Without it, closing the laptop lid eventually kills the session; WITH it, worst case the VM
recycles and you rerun this notebook tomorrow and it resumes from the last checkpoint.

## 1. Config — paste your HF token, then Run All

In [ ]:
HF_TOKEN = "paste-your-hf-token-here"  # write-scope token for timthy45 (uploads checkpoints)

DATA_REPO = "timthy45/pnid-extraction-datasets"     # private dataset repo (already uploaded)
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"     # private model repo for checkpoints (auto-created)
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

N_EPOCHS = 3
LR = 1e-4
CHECKPOINT_EVERY_N_STEPS = 200
TIME_BUDGET_HOURS = 11          # stop cleanly (with checkpoint) before Colab can kill the runtime
MAX_KAGGLE_EXAMPLES = 1000      # keep pool balanced, not Kaggle-dominated
MAX_RELATION_SHEETS = 250       # PID2Graph 1500px patches to sample pairs from (1629 available)
RELATION_PAIRS_PER_SHEET = 12   # half connected, half not

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here", "Paste your real HF token above before running"

## 2. Install deps + GPU check

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q peft pytesseract huggingface_hub

import torch
assert torch.cuda.is_available(), "No GPU - set Runtime type to GPU before continuing"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 3. Pull all 3 datasets from HF Hub → local VM disk

No Drive, no Google auth. ~17GB download lands on fast local NVMe; HF CDN into Colab is quick.
Re-running is cheap (HF caches the zips).

In [ ]:
import zipfile, time
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)

t0 = time.time()
for fname, extract_to in [
    ("gupta_pid/PID_Dataset.zip", DATA / "gupta"),
    ("kaggle_pid_symbols/kaggle_pid_symbols.zip", DATA / "kaggle"),
    ("pid2graph/PID2Graph.zip", DATA / "pid2graph"),
]:
    if extract_to.exists() and any(extract_to.iterdir()):
        print(f"{extract_to} already extracted, skipping")
        continue
    zp = hf_hub_download(repo_id=DATA_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    print(f"downloaded {fname} ({time.time()-t0:.0f}s), extracting...")
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(extract_to)
    print(f"extracted -> {extract_to} ({time.time()-t0:.0f}s)")

GUPTA_RAW = DATA / "gupta" / "PID_Dataset" / "0__raw_data"
KAGGLE_IMAGES = DATA / "kaggle" / "archive" / "images (3)"
KAGGLE_LABELS = DATA / "kaggle" / "archive" / "labels (2)"
PID2GRAPH_OPEN100 = DATA / "pid2graph" / "PID2Graph" / "Patched" / "PID2Graph OPEN100"

for p in [GUPTA_RAW / "sheets" / "train", KAGGLE_IMAGES, KAGGLE_LABELS, PID2GRAPH_OPEN100]:
    assert p.exists(), f"expected path missing after extract: {p}"
print("All dataset paths verified.")
print("Gupta train sheets:", len(list((GUPTA_RAW / "sheets" / "train").glob("*.*"))))
print("Kaggle images:", len(list(KAGGLE_IMAGES.glob("*.jpg"))))
print("PID2Graph OPEN100 patch graphml:", len(list(PID2GRAPH_OPEN100.rglob("*.graphml"))))

## 4. Kaggle 32-class name map (inlined — no Drive `classes.json` dependency)

Copied verbatim from `Stage4_Phase2_Data_Preparation.ipynb` (checklist 2.2). Name source is the
dataset author's GitHub `dataset.yaml` — unverified against an official listing, flagged there.

In [ ]:
CLASS_NAMES = {
    "0": "Not_used", "1": "Gate_Valve", "2": "Ball_Valve", "3": "Globe_valve_NO",
    "4": "Gate_valve_NO", "5": "Globe_valve_NO", "6": "Butterfly_valve", "7": "Plug_valve",
    "8": "Check_valve", "9": "Diaphragm_valve", "10": "Needle_valve",
    "11": "Half_Filled_Gate_Valve", "12": "Gate_Valve_NC", "13": "Globle_valve_NC",
    "14": "Control_Valve", "15": "Rotary_Valve", "16": "Ball_valve_NC", "17": "Paddle_blind",
    "18": "Spectacle_blind_Closed", "19": "Spectacle_blind_Open", "20": "Reducer",
    "21": "Flange_or_Nozzle", "22": "Rupture_disk", "23": "Pipe_Insulation_or_Tracing",
    "24": "Flow_Arrow", "25": "sight_glass", "26": "Instrument_Field", "27": "Instrument_Field",
    "28": "Instrument_Panel", "29": "Instrument_Aux_Panel", "30": "box",
    "31": "Instrument_Panel", "32": "box",
}

## 5. Build the 4-task mixed training pool

Tasks 1-3 are the proven builders from `Stage4_DomainAdapt_OCR_Training.ipynb` (paths adapted to
the raw zip layouts). Task 4 (connectivity) is new, built from PID2Graph OPEN100 graphml.

In [ ]:
import json, random
import xml.etree.ElementTree as ET
import pytesseract
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append((x0, y0, x1, y1))
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return parts[0], [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def boxes_intersect(a, b):
    return a[0] < b[2] and a[2] > b[0] and a[1] < b[3] and a[3] > b[1]

def ocr_tags(tile_img, min_len=2):
    """Real Tesseract OCR - pseudo-label, not verified ground truth."""
    tokens = [t.strip() for t in pytesseract.image_to_string(tile_img).split() if len(t.strip()) >= min_len]
    return sorted(set(t for t in tokens if any(c.isdigit() for c in t) or "-" in t))

OCR_PROMPT = "List every text tag or label you can read in this P&ID tile."
COUNT_PROMPT = "How many distinct symbols (valves, instruments, fittings, etc.) are visible in this P&ID tile?"
TYPED_SUMMARY_PROMPT = "What types of symbols are visible in this P&ID tile, and how many of each?"

def build_gupta_examples():
    examples = []
    for sheet_path in sorted((GUPTA_RAW / "sheets" / "train").glob("*.*")):
        label_path = GUPTA_RAW / "labels" / "train" / f"{sheet_path.stem}.txt"
        img = Image.open(sheet_path).convert("RGB")
        W, H = img.size
        orig_boxes = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    orig_boxes.append(yolo_line_to_xyxy(line, W, H)[1])
        for tbox in compute_tile_grid(W, H):
            n_boxes = sum(1 for b in orig_boxes if boxes_intersect(b, tbox))
            tags = ocr_tags(img.crop(tbox))
            if tags:
                examples.append({"task": "ocr", "image_path": str(sheet_path), "crop": list(tbox),
                                 "prompt": OCR_PROMPT,
                                 "target_text": "Tags visible in this tile: " + ", ".join(tags)})
            count_target = (f"There are {n_boxes} symbols visible in this tile." if n_boxes != 1
                            else "There is 1 symbol visible in this tile.")
            examples.append({"task": "count", "image_path": str(sheet_path), "crop": list(tbox),
                             "prompt": COUNT_PROMPT, "target_text": count_target})
    return examples

def build_kaggle_examples(max_images):
    from collections import Counter
    examples = []
    for lbl in sorted(KAGGLE_LABELS.glob("*.txt"))[:max_images]:
        lines = [l for l in lbl.read_text().splitlines() if l.strip()]
        if not lines:
            continue
        img_path = KAGGLE_IMAGES / f"{lbl.stem}.jpg"
        if not img_path.exists():
            continue
        counts = Counter(l.split()[0] for l in lines)
        parts = [f"{CLASS_NAMES.get(cid, 'class_' + cid)} x{n}" for cid, n in counts.most_common()]
        examples.append({"task": "typed_summary", "image_path": str(img_path), "crop": None,
                         "prompt": TYPED_SUMMARY_PROMPT,
                         "target_text": "This tile contains: " + ", ".join(parts) + "."})
    return examples

# --- Task 4: connectivity Q&A from PID2Graph OPEN100 graph edges (real ground truth) ---
RELATION_PROMPT_TEMPLATE = (
    "In this P&ID crop there is a symbol at [{ax0}, {ay0}, {ax1}, {ay1}] and another at "
    "[{bx0}, {by0}, {bx1}, {by1}] (pixel coordinates, top-left origin). "
    "Are these two symbols directly connected to each other?"
)
MAX_PAIR_SPAN_PX = 1400   # skip pairs whose union crop would be huge (far-apart symbols)
CROP_MARGIN = 80

def parse_graphml(path):
    """Returns ({node_id: [x0,y0,x1,y1]}, set of frozenset edges). Maps data keys via attr.name."""
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {}
        for d in node.findall("g:data", ns):
            vals[keymap.get(d.get("key"), "")] = d.text
        try:
            box = [float(vals["xmin"]), float(vals["ymin"]), float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
        nodes[node.get("id")] = box
    for edge in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        s, t = edge.get("source"), edge.get("target")
        if s in nodes and t in nodes:
            edges.add(frozenset((s, t)))
    return nodes, edges

def build_relation_examples(max_sheets, pairs_per_sheet, seed=0):
    rng = random.Random(seed)
    examples = []
    # Patched tree: per-sheet folders of 1500x1500 patch graphml/png pairs (1629 total).
    # Coords are patch-local; a rare 1-2px bbox overshoot exists in the data - crops clamp anyway.
    gml_files = sorted(PID2GRAPH_OPEN100.rglob("*.graphml"))
    rng.shuffle(gml_files)
    gml_files = gml_files[:max_sheets]
    for gml in gml_files:
        png = gml.with_suffix(".png")
        if not png.exists():
            continue
        try:
            nodes, edges = parse_graphml(gml)
        except ET.ParseError:
            continue
        if len(nodes) < 4 or not edges:
            continue
        node_ids = list(nodes)
        pos_pool = [tuple(e) for e in edges]
        rng.shuffle(pos_pool)
        neg_pool = []
        attempts = 0
        while len(neg_pool) < pairs_per_sheet and attempts < 200:
            attempts += 1
            a, b = rng.sample(node_ids, 2)
            if frozenset((a, b)) not in edges:
                neg_pool.append((a, b))
        for pair, connected in [(p, True) for p in pos_pool[:pairs_per_sheet // 2]] +                                [(p, False) for p in neg_pool[:pairs_per_sheet // 2]]:
            a, b = pair
            ba, bb = nodes[a], nodes[b]
            ux0, uy0 = min(ba[0], bb[0]), min(ba[1], bb[1])
            ux1, uy1 = max(ba[2], bb[2]), max(ba[3], bb[3])
            if ux1 - ux0 > MAX_PAIR_SPAN_PX or uy1 - uy0 > MAX_PAIR_SPAN_PX:
                continue
            cx0, cy0 = max(0, int(ux0 - CROP_MARGIN)), max(0, int(uy0 - CROP_MARGIN))
            cx1, cy1 = int(ux1 + CROP_MARGIN), int(uy1 + CROP_MARGIN)
            prompt = RELATION_PROMPT_TEMPLATE.format(
                ax0=int(ba[0] - cx0), ay0=int(ba[1] - cy0), ax1=int(ba[2] - cx0), ay1=int(ba[3] - cy0),
                bx0=int(bb[0] - cx0), by0=int(bb[1] - cy0), bx1=int(bb[2] - cx0), by1=int(bb[3] - cy0),
            )
            target = ("Yes - these two symbols are directly connected." if connected
                      else "No - these two symbols are not directly connected.")
            examples.append({"task": "relation", "image_path": str(png), "crop": [cx0, cy0, cx1, cy1],
                             "prompt": prompt, "target_text": target})
    return examples

print("Building Gupta OCR+count examples (Tesseract over all 72 sheets - takes several minutes on local disk)...")
t0 = time.time()
gupta_examples = build_gupta_examples()
print(f"  {len(gupta_examples)} examples in {time.time()-t0:.0f}s")

kaggle_examples = build_kaggle_examples(MAX_KAGGLE_EXAMPLES)
print(f"  {len(kaggle_examples)} Kaggle typed-summary examples")

relation_examples = build_relation_examples(MAX_RELATION_SHEETS, RELATION_PAIRS_PER_SHEET)
print(f"  {len(relation_examples)} PID2Graph relation examples")

all_examples = gupta_examples + kaggle_examples + relation_examples
random.seed(0)
random.shuffle(all_examples)
from collections import Counter
print("\nTotal pool:", len(all_examples), dict(Counter(e["task"] for e in all_examples)))

## 6. Load Qwen3-VL-8B + LoRA (r=16, all-linear — same recipe as `train.py`)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, get_peft_model

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
print("Base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Checkpoint repo + auto-resume

Checkpoints live in a private HF model repo — they survive VM death. On rerun, this cell pulls
the latest adapter + `progress.json` and continues at the exact epoch/step.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=CKPT_REPO, repo_type="model", private=True, exist_ok=True)

CKPT_LOCAL = Path("/content/checkpoints")
LATEST_DIR = CKPT_LOCAL / "latest"
PROGRESS_FILE = LATEST_DIR / "progress.json"
CKPT_LOCAL.mkdir(exist_ok=True)

start_epoch, start_step = 0, 0
try:
    snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                      allow_patterns=["latest/*"], local_dir=str(CKPT_LOCAL))
except Exception as e:
    print("no remote snapshot pulled:", type(e).__name__)

if PROGRESS_FILE.exists() and (LATEST_DIR / "adapter_model.safetensors").exists():
    progress = json.loads(PROGRESS_FILE.read_text())
    start_epoch, start_step = progress["epoch"], progress["step"] + 1
    model.load_adapter(str(LATEST_DIR), adapter_name="default", is_trainable=True)
    print(f"RESUMED from HF checkpoint: epoch {start_epoch}, step {start_step}")
else:
    print("No checkpoint found - starting fresh from epoch 0 step 0")

def save_and_push_checkpoint(epoch, step, also_epoch_snapshot=False):
    """Local save + push to HF. Push failures are logged, never fatal - training continues."""
    model.save_pretrained(str(LATEST_DIR))
    PROGRESS_FILE.write_text(json.dumps({"epoch": epoch, "step": step}))
    try:
        api.upload_folder(folder_path=str(LATEST_DIR), path_in_repo="latest",
                          repo_id=CKPT_REPO, repo_type="model")
        if also_epoch_snapshot:
            api.upload_folder(folder_path=str(LATEST_DIR), path_in_repo=f"epoch_{epoch}",
                              repo_id=CKPT_REPO, repo_type="model")
    except Exception as e:
        print(f"  [warn] HF push failed ({type(e).__name__}: {e}) - local checkpoint still saved, will retry next interval")

## 8. Training loop — run this and walk away

Checkpoints every 200 steps (local + HF push). Time-budget stop before Colab's hard limit.
Per-step failures (corrupt image, transient OOM) are skipped, not fatal. If the runtime dies
anyway: reopen tomorrow, Run All, it resumes from the last pushed checkpoint.

In [ ]:
import gc

def build_labeled_inputs(image, prompt, target_text, image_crop=None):
    if image_crop is not None:
        image = image.crop(image_crop)
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    prompt_len = prompt_inputs["input_ids"].shape[1]
    full = processor.apply_chat_template(
        messages + [{"role": "assistant", "content": [{"type": "text", "text": target_text}]}],
        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    labels = full["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full["labels"] = labels
    return {k: v.to(model.device) for k, v in full.items()}

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
run_t0 = time.time()
time_budget_s = TIME_BUDGET_HOURS * 3600
stopped_for_time = False

for epoch in range(start_epoch, N_EPOCHS):
    random.seed(epoch)
    order = list(range(len(all_examples)))
    random.shuffle(order)
    skip_to = start_step if epoch == start_epoch else 0

    epoch_losses = {"ocr": [], "count": [], "typed_summary": [], "relation": []}
    t0 = time.time()
    for step, idx in enumerate(order):
        if step < skip_to:
            continue
        if time.time() - run_t0 > time_budget_s:
            save_and_push_checkpoint(epoch, step - 1)
            print(f"TIME BUDGET REACHED ({TIME_BUDGET_HOURS}h) - checkpointed at epoch {epoch} step {step-1}. "
                  "Rerun the notebook to continue from here.")
            stopped_for_time = True
            break
        ex = all_examples[idx]
        try:
            img = Image.open(ex["image_path"]).convert("RGB")
            inputs = build_labeled_inputs(img, ex["prompt"], ex["target_text"], image_crop=ex.get("crop"))
            out = model(**inputs)
            loss = out.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            epoch_losses[ex["task"]].append(loss.item())
        except torch.cuda.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True)
            gc.collect(); torch.cuda.empty_cache()
            print(f"  [skip] step {step} OOM ({ex['task']}) - cache cleared, continuing")
            continue
        except Exception as e:
            print(f"  [skip] step {step} ({ex['task']}) failed: {type(e).__name__}: {e}")
            continue

        if step % 50 == 0:
            recent = [l for ls in epoch_losses.values() for l in ls][-50:]
            avg = sum(recent) / len(recent) if recent else float("nan")
            print(f"epoch {epoch} step {step}/{len(order)} avg_loss={avg:.4f} elapsed={time.time()-t0:.0f}s")
        if step % CHECKPOINT_EVERY_N_STEPS == 0 and step > 0:
            save_and_push_checkpoint(epoch, step)
            print(f"  [checkpoint] epoch {epoch} step {step} (local + HF)")

    if stopped_for_time:
        break
    save_and_push_checkpoint(epoch, len(order) - 1, also_epoch_snapshot=True)
    print(f"=== epoch {epoch} done (snapshot pushed to HF as epoch_{epoch}/) ===")
    for task, losses in epoch_losses.items():
        if losses:
            print(f"  {task:15s} n={len(losses):5d} mean_loss={sum(losses)/len(losses):.4f}")
    start_step = 0

if not stopped_for_time:
    print("\nTraining complete - final adapter in HF repo:", CKPT_REPO)

## 9. After training — quick smoke test

Loads nothing new — just generates from the trained model on one example per task to eyeball
that it learned the domain (not a benchmark; the per-stage benchmarks come later).

In [ ]:
model.eval()
by_task = {}
for ex in all_examples:
    by_task.setdefault(ex["task"], ex)
with torch.no_grad():
    for task, ex in by_task.items():
        img = Image.open(ex["image_path"]).convert("RGB")
        if ex.get("crop"):
            img = img.crop(ex["crop"])
        messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": ex["prompt"]}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"=== {task} ===")
        print("target:", ex["target_text"][:150])
        print("model :", text[:150])
        print()